In [1]:
import sys,os
sys.path.append(os.getcwd())
from src.utils.data_utils import round_to_closest_indices, make_padding
import logging
import numpy as np
import pandas as pd
import torch
from omegaconf import DictConfig, OmegaConf
import yaml
import hashlib
from numpy.lib.stride_tricks import sliding_window_view
from datetime import datetime
import time
import polars
import gc
from functools import partial
import glob 
from pytorch_lightning.utilities import rank_zero_only
import pickle

/home/a.galliamov/miniconda3/envs/wind/lib/python3.10/site-packages/lightning_fabric/__init__.py:36: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __import__("pkg_resources").declare_namespace(__name__)


In [8]:
import hydra

@hydra.main(version_base=None, config_path=os.path.join(os.getcwd(),"configs"), config_name="cmip5_TestNet")
def load_climate_data(self, cfg: DictConfig):
    self.cfg = cfg 
    dtype = np.float16 if self.cfg.process.precision == 16 else np.float32
    self.time_coords = np.load(os.path.join(self.cfg.train.data_dir, 'time.npy')).astype('datetime64[D]')
    self.lat_coords = np.load(os.path.join(self.cfg.train.data_dir, 'lat.npy'))
    self.lon_coords = np.load(os.path.join(self.cfg.train.data_dir, 'lon.npy'))

    var_data = np.empty(
        (len(self.cfg.train.variables), len(self.time_coords), len(self.lat_coords), len(self.lon_coords)),
        dtype=dtype)
    for i, var in enumerate(self.cfg.train.variables):
        var_data[i] = np.load(os.path.join(self.cfg.train.data_dir, var + f'_{self.cfg.process.precision}.npy'))
    logging.info(f"CMIP data loaded {var_data.shape}")
    # var_data_torch = torch.from_numpy(var_data).type(torch.float32)

    return var_data

In [ ]:
config_path=os.path.join(os.getcwd(),"configs")
config_path

'/home/a.galliamov/storm_prediction/configs/cmip5_TestNet.yaml'

In [12]:
def get_rundir_name():
    now = datetime.now()
    return f'out/{now:%Y-%m-%d}/{now:%H-%M-%S}'

cfg_path = os.path.join(os.getcwd(), "configs", "cmip5_TestNet.yaml")

# 2) Загрузите cfg
cfg = OmegaConf.load(cfg_path)

rundir = get_rundir_name()
os.makedirs(rundir, exist_ok=True)
os.chdir(rundir) 

In [14]:
var = load_climate_data()
var.shape

usage: ipykernel_launcher.py [--help] [--hydra-help] [--version]
                             [--cfg {job,hydra,all}] [--resolve]
                             [--package PACKAGE] [--run] [--multirun]
                             [--shell-completion] [--config-path CONFIG_PATH]
                             [--config-name CONFIG_NAME]
                             [--config-dir CONFIG_DIR]
                             [--experimental-rerun EXPERIMENTAL_RERUN]
                             [--info [{all,config,defaults,defaults-tree,plugins,searchpath}]]
                             [overrides ...]
ipykernel_launcher.py: error: unrecognized arguments: --f=/run/user/4200126/jupyter/runtime/kernel-v31488aae1e0984e5928efb168f7dddadca5430fc1.json


SystemExit: 2

In [ ]:
почему сразу переходит в torch.float32? Где обратная нормализация?